# 03. Inspect the saved model-ready data

This notebook reads `train_sequences.npz` and its human-readable CSV export. It
shows the first two sequences and the final sequence, including every padded
position, with chord labels, note tensors and masks side by side. It does not
refit encoders, regenerate preprocessing outputs, change CSVs or train a model.

Select the existing project `.venv` kernel and use **Restart Kernel and Run All**.
Paths work from the notebooks folder or another directory within the repository.
The saved artifact is a single-track development baseline with synthetic basics
and recognized reference pseudo-labels; this inspection is not model evaluation.

In [1]:
from pathlib import Path
import csv
import hashlib
import html
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents)
           if (p/'experiments/rahul/src/transformer_dataset.py').exists()),None)
if ROOT is None:
    raise RuntimeError('Run from within the vocal-harmonization repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from experiments.rahul.src.transformer_dataset import SavedSequenceDataset
from experiments.rahul.src.encode_features import FeatureEncoder, CATEGORICAL

OUTPUT=ROOT/'experiments/rahul/outputs/model_preprocessing'
NPZ_PATH=OUTPUT/'train_sequences.npz'
CSV_PATH=OUTPUT/'train_sequences_readable.csv'
schema=json.loads((OUTPUT/'train_sequences_readable_schema.json').read_text(encoding='utf-8'))
export_report=json.loads((OUTPUT/'train_sequences_readable_validation.json').read_text(encoding='utf-8'))
with np.load(NPZ_PATH,allow_pickle=False) as archive:
    arrays={k:archive[k] for k in archive.files}
with CSV_PATH.open(encoding='utf-8',newline='') as handle:
    reader=csv.DictReader(handle)
    csv_columns=reader.fieldnames
    records=list(reader)
frame=pd.DataFrame(records)  # Preserve original CSV text: no automatic NA/numeric conversions.
encoder=FeatureEncoder.load(OUTPUT)
dataset=SavedSequenceDataset(NPZ_PATH)
N,L=arrays['beat_indices'].shape
M=arrays['inputs__melody_pitch_ids'].shape[-1]
chosen_sequences=list(dict.fromkeys([0,1,N-1]))

def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
protected_before={path:sha(ROOT/path) for path in export_report['source_sha256']}
new_export_before={p.name:sha(p) for p in OUTPUT.glob('train_sequences_readable*')}
pd.set_option('display.max_rows',60)
pd.set_option('display.max_colwidth',None)

def side_by_side(left,right,left_title,right_title):
    def table(df,title):
        return '<div style="min-width:0;overflow-x:auto"><h4>'+html.escape(title)+'</h4>'+df.to_html(index=False,escape=True)+'</div>'
    display(HTML('<div style="display:grid;grid-template-columns:1fr 1fr;gap:18px;align-items:start">'
                 +table(left,left_title)+table(right,right_title)+'</div>'))

print('Source:',NPZ_PATH.relative_to(ROOT).as_posix())
print('Readable export:',CSV_PATH.relative_to(ROOT).as_posix())
print('Saved feature encoders fitted on tracks:',encoder.training_tracks)

Source: experiments/rahul/outputs/model_preprocessing/train_sequences.npz
Readable export: experiments/rahul/outputs/model_preprocessing/train_sequences_readable.csv
Saved feature encoders fitted on tracks: ['382']


## 1. Verify the actual archive and row structure

`sequence_id` and `position_in_sequence` are zero-based. Each row is one stored
position, including padding. `track_id` repeats the sequence's stored string ID.
Metadata `beat_index` is the exact int64 source beat ID, with -1 for padding;
`inputs__beat_index` separately preserves the floating model input, which is 0
at padding. They serve different purposes and are both retained.

Every `inputs__...` or `targets__...` column has exactly the original NPZ array
name. Melody arrays are JSON lists, and timing is a nested list of shape M by 3.
Sequence-level genre vectors repeat unchanged at all positions, including padding.
The dimensions and counts below come from the actual archive, not fixed constants.

In [2]:
display(pd.DataFrame([{'NPZ_array':k,'shape':list(v.shape),'dtype':str(v.dtype)} for k,v in arrays.items()]))
real=arrays['inputs__beat_attention_mask']
loss=arrays['targets__target_loss_mask']
counts=pd.DataFrame([{
    'sequences':N,'sequence_length':L,'expected_rows_from_shape':N*L,'CSV_rows':len(records),
    'CSV_columns':len(csv_columns),'real_positions':int(real.sum()),
    'padded_positions':int((~real).sum()),'eligible_targets':int(loss.sum()),
    'invalid_real_targets':int((real & ~loss).sum()),'note_capacity':M,
}])
display(counts)
assert len(records)==N*L
assert csv_columns==list(schema['columns'])
assert {k for k in arrays if k.startswith(('inputs__','targets__'))}<=set(csv_columns)
print('All numerical input, target and mask arrays have a corresponding CSV column.')

,NPZ_array,shape,dtype
0,inputs__basic_chord_ids,"[54, 8]",int64
1,inputs__basic_root_ids,"[54, 8]",int64
2,inputs__basic_quality_ids,"[54, 8]",int64
3,inputs__previous_basic_chord_ids,"[54, 8]",int64
4,inputs__next_basic_chord_ids,"[54, 8]",int64
5,inputs__key_ids,"[54, 8]",int64
6,inputs__mode_ids,"[54, 8]",int64
7,inputs__key_confidence_ids,"[54, 8]",int64
8,inputs__roman_ids,"[54, 8]",int64
9,inputs__harmonic_function_ids,"[54, 8]",int64


,sequences,sequence_length,expected_rows_from_shape,CSV_rows,CSV_columns,real_positions,padded_positions,eligible_targets,invalid_real_targets,note_capacity
0,54,8,432,432,49,428,4,425,3,4


All numerical input, target and mask arrays have a corresponding CSV column.


## 2. First two sequences and the final padded sequence

The left tables show basic-chord IDs and decoded input context. The right tables
show separate supervision. Labels come from the saved vocabulary, not new music
analysis. PAD, UNK, N, START and END remain distinct.

A target ID of -100 has no class label: the readable column says **IGNORE**.
The original rejected reference label cannot be recovered from that ID, so this
export does not invent it. Both invalid and padded targets are ignored; the real
beat mask and `is_padding` distinguish them. These target columns are inspection
data, not additional inputs to a future model.

In [3]:
for s in chosen_sequences:
    selected=frame[frame['sequence_id']==str(s)]
    sample=dataset[s]
    assert selected['beat_index'].astype(int).tolist()==sample['metadata']['beat_indices'].tolist()
    display(Markdown(f"### Sequence {s}; track {sample['metadata']['track_id']}"))
    left=selected[['position_in_sequence','beat_index','is_padding','inputs__basic_chord_ids',
                   'basic_chord_decoded','previous_basic_chord_decoded','next_basic_chord_decoded']].rename(columns={
        'position_in_sequence':'pos','beat_index':'source beat','is_padding':'padding',
        'inputs__basic_chord_ids':'basic ID','basic_chord_decoded':'basic',
        'previous_basic_chord_decoded':'previous','next_basic_chord_decoded':'next'})
    right=selected[['position_in_sequence','targets__target_chord_ids','reference_chord_decoded',
                    'inputs__beat_attention_mask','targets__target_loss_mask']].rename(columns={
        'position_in_sequence':'pos','targets__target_chord_ids':'target ID',
        'reference_chord_decoded':'reference','inputs__beat_attention_mask':'real beat',
        'targets__target_loss_mask':'loss mask'})
    side_by_side(left,right,'Chord inputs','Separate targets and beat masks')

### Sequence 0; track 382

### Sequence 1; track 382

### Sequence 53; track 382

## 3. Melody timing tensors and masks side by side

All note slots are preserved. Pitch tokens use MIDI+2 after rounding, and pitch
class tokens use pitch class+2; 0 is PAD and 1 is UNK. The separate continuous
`inputs__melody_pitch_values` holds original fractional MIDI/127 at saved precision.

Each timing row is **[whole note duration / beat duration, original relative onset
/ beat duration, overlap / beat duration]**. Negative onsets are valid for a note
that began before this beat. The display below uses six decimal places only for
compact reading; the CSV retains the full stored values, verified in Section 6.

`melody_note_mask` marks real notes. `melody_attention_mask` additionally enables
one zero-data dummy slot on silence or beat padding. `melody_silent` flags those
rows. The dummy is not a real MIDI note. For later PyTorch key-padding masks,
invert the safe keep mask and zero silent pooled melody representations.

In [4]:
for s in chosen_sequences:
    selected=[r for r in records if int(r['sequence_id'])==s]
    note_table=pd.DataFrame([{
        'pos':r['position_in_sequence'],'MIDI tokens':r['inputs__melody_pitch_ids'],
        'decoded rounded MIDI':r['melody_midi_rounded_decoded'],
        'note mask':r['inputs__melody_note_mask'],'safe attention':r['inputs__melody_attention_mask'],
        'silent':r['inputs__melody_silent'],
    } for r in selected])
    timing_table=pd.DataFrame([{
        'pos':r['position_in_sequence'],
        'timing [duration, onset, overlap]':np.array2string(
            np.asarray(json.loads(r['inputs__melody_timing'])),precision=6,floatmode='maxprec_equal'),
    } for r in selected])
    # Preformatted timing matrices preserve line breaks without changing the CSV values.
    timing_html=timing_table.to_html(index=False,escape=True).replace('<td>','<td style="white-space:pre;font-family:monospace">')
    timing_html=timing_html.replace(chr(92)+'n','<br>')
    display(Markdown(f'### Melody in sequence {s}'))
    display(HTML('<div style="display:grid;grid-template-columns:1.4fr 1fr;gap:18px;align-items:start">'
        +'<div style="overflow-x:auto"><h4>Pitch slots and independent masks</h4>'+note_table.to_html(index=False,escape=True)+'</div>'
        +'<div style="overflow-x:auto"><h4>Exact tensor structure (display rounded)</h4>'+timing_html+'</div></div>'))

### Melody in sequence 0

### Melody in sequence 1

### Melody in sequence 53

## 4. Saved musical context and genre coordinates

The categorical IDs below decode with the existing saved feature vocabularies.
Different categorical fields have their own mappings; identical IDs across two
fields do not imply the same musical meaning. Current/previous/next chords share
the basic vocabulary. Genre is a sequence-level multi-hot vector in the displayed
saved order. Its presence on a padded beat is intentional and faithful to the NPZ.
Bar indices remain zero-based numerical values, and missing bar/beat-position
information is described by `inputs__beat_timing_known_mask`.

In [5]:
first_real=next(r for r in records if not json.loads(r['is_padding']) and r['basic_chord_decoded']!='N')
context_rows=[]
for tensor,field in CATEGORICAL.items():
    context_rows.append({'input tensor':tensor,'saved ID':int(first_real['inputs__'+tensor]),
                         'decoded token':first_real[tensor.removesuffix('_ids')+'_decoded'],
                         'vocabulary':field})
display(pd.DataFrame(context_rows))
display(pd.DataFrame({'genre coordinate':range(len(encoder.genres)),'saved label':encoder.genres}))
display(frame[frame['sequence_id']==str(N-1)][[
    'position_in_sequence','beat_index','is_padding','inputs__genre_features','genre_labels_decoded',
    'inputs__bar_number','inputs__relative_bar_number','inputs__beat_position',
    'inputs__event_duration','inputs__beat_timing_known_mask']])

,input tensor,saved ID,decoded token,vocabulary
0,basic_chord_ids,16,F,basic_chord_canonical
1,basic_root_ids,14,F,basic_root
2,basic_quality_ids,7,maj,basic_quality
3,previous_basic_chord_ids,2,N,previous_basic_chord
4,next_basic_chord_ids,16,F,next_basic_chord
5,key_ids,6,C,key
6,mode_ids,6,major,mode
7,key_confidence_ids,6,high,key_confidence
8,roman_ids,7,IV,roman_numeral
9,harmonic_function_ids,7,predominant,harmonic_function


,genre coordinate,saved label
0,0,UNK
1,1,classical


,position_in_sequence,beat_index,is_padding,inputs__genre_features,genre_labels_decoded,inputs__bar_number,inputs__relative_bar_number,inputs__beat_position,inputs__event_duration,inputs__beat_timing_known_mask
424,0,424,false,"[0,1]","[""classical""]",106,0,1,0.5600000023841858,"[true,true]"
425,1,425,false,"[0,1]","[""classical""]",106,0,2,0.5600000023841858,"[true,true]"
426,2,426,false,"[0,1]","[""classical""]",106,0,3,0.5600000023841858,"[true,true]"
427,3,427,false,"[0,1]","[""classical""]",106,0,4,0.5199999809265137,"[true,true]"
428,4,-1,true,"[0,1]","[""classical""]",0,0,0,0,"[false,false]"
429,5,-1,true,"[0,1]","[""classical""]",0,0,0,0,"[false,false]"
430,6,-1,true,"[0,1]","[""classical""]",0,0,0,0,"[false,false]"
431,7,-1,true,"[0,1]","[""classical""]",0,0,0,0,"[false,false]"


## 5. Meaning of every CSV column

The complete data dictionary is also saved as
`train_sequences_readable_columns.md` beside the CSV, with a machine-readable
schema in `train_sequences_readable_schema.json`. The table below includes every
column, its role, source and representation. All booleans use JSON true/false;
nested lists should be read using `json.loads`, never `eval`.

In [6]:
display(pd.DataFrame([{
    'column':name,'role':spec['role'],
    'source':spec.get('npz_array',spec.get('derived_from','sequence metadata')),
    'dtype':spec.get('dtype','derived'),'shape per row':spec.get('per_row_shape','scalar/decoded'),
    'serialization':spec['serialization'],'meaning':spec['meaning'],
} for name,spec in schema['columns'].items()]))

,column,role,source,dtype,shape per row,serialization,meaning
0,sequence_id,identity,sequence metadata,derived,scalar/decoded,integer,Zero-based archive sequence index; equals SavedSequenceDataset item index.
1,position_in_sequence,identity,sequence metadata,derived,scalar/decoded,integer,Zero-based position within the stored sequence.
2,track_id,metadata,track_ids,<U3,scalar/decoded,text,"Exact track_ids value, repeated for all positions in the sequence."
3,beat_index,metadata,beat_indices,int64,scalar/decoded,integer,Exact int64 source beat_indices value; -1 at padded positions. Distinct from the floating input tensor.
4,is_padding,derived inspection,sequence metadata,derived,scalar/decoded,JSON boolean,Logical inverse of inputs__beat_attention_mask; true marks a nonexistent beat.
5,inputs__basic_chord_ids,inputs,inputs__basic_chord_ids,int64,[],JSON scalar,Canonical basic-chord category ID from the saved shared basic vocabulary.
6,inputs__basic_root_ids,inputs,inputs__basic_root_ids,int64,[],JSON scalar,"Basic root-spelling category ID; genuine N roots use N, padded positions use PAD."
7,inputs__basic_quality_ids,inputs,inputs__basic_quality_ids,int64,[],JSON scalar,Basic-quality category ID (major/minor/diminished/N or reserved token).
8,inputs__previous_basic_chord_ids,inputs,inputs__previous_basic_chord_ids,int64,[],JSON scalar,Previous basic chord ID in the shared vocabulary; START at track beginning.
9,inputs__next_basic_chord_ids,inputs,inputs__next_basic_chord_ids,int64,[],JSON scalar,Next basic chord ID in the shared vocabulary; END at track end.


## 6. Reconstruct and verify every original NPZ array

This check parses the actual exported CSV, casts numerical values back to their
saved dtypes, restores sequence and position axes, and compares **every array's
bytes** to the original. Each repeated genre vector is checked independently.
The metadata arrays are reconstructed too. Full precision matters: convenient
display rounding in a table is not allowed to change the export.

Decoded columns are separately checked against the loaded vocabularies and pitch
token rules. The original NPZ, vocabularies, pre-existing CSVs, modules and prior
notebooks are hash-checked. This inspection never writes or overwrites them.

In [7]:
restored={k:np.empty_like(v) for k,v in arrays.items()}
for index,row in enumerate(records):
    s,p=divmod(index,L)
    assert (int(row['sequence_id']),int(row['position_in_sequence']))==(s,p)
    assert row['track_id']==str(arrays['track_ids'][s])
    assert int(row['beat_index'])==arrays['beat_indices'][s,p]
    restored['track_ids'][s]=row['track_id']
    restored['beat_indices'][s,p]=int(row['beat_index'])
    for key,array in arrays.items():
        if key in ('track_ids','beat_indices'): continue
        value=np.asarray(json.loads(row[key]),dtype=array.dtype)
        expected=array[s] if key=='inputs__genre_features' else array[s,p]
        assert value.shape==expected.shape and value.tobytes()==expected.tobytes(),(s,p,key)
        if key=='inputs__genre_features': restored[key][s]=value
        else: restored[key][s,p]=value
    assert json.loads(row['is_padding'])==(not bool(arrays['inputs__beat_attention_mask'][s,p]))
    for tensor,field in CATEGORICAL.items():
        token=encoder.vocabs[field].decode(int(row['inputs__'+tensor]))
        assert row[tensor.removesuffix('_ids')+'_decoded']==token
    target=int(row['targets__target_chord_ids'])
    assert row['reference_chord_decoded']==('IGNORE' if target==-100 else encoder.targets.decode(target))
    assert json.loads(row['melody_sources_decoded'])==[
        encoder.vocabs['melody_sources'].decode(v) for v in json.loads(row['inputs__melody_source_ids'])]
    for ids_column,decoded_column in [('inputs__melody_pitch_ids','melody_midi_rounded_decoded'),
                                      ('inputs__melody_pitch_class_ids','melody_pitch_classes_decoded')]:
        assert json.loads(row[decoded_column])==[
            'PAD' if v==0 else 'UNK' if v==1 else v-2 for v in json.loads(row[ids_column])]
    assert json.loads(row['genre_vocabulary'])==encoder.genres
    assert json.loads(row['genre_labels_decoded'])==[
        genre for genre,value in zip(encoder.genres,json.loads(row['inputs__genre_features'])) if value!=0]

array_checks=[]
for key,array in arrays.items():
    match=(restored[key].shape==array.shape and restored[key].dtype==array.dtype
           and restored[key].tobytes()==array.tobytes())
    assert match,key
    array_checks.append({'NPZ array':key,'shape':list(array.shape),'dtype':str(array.dtype),'byte-identical':match})
display(pd.DataFrame(array_checks))
assert all(sha(ROOT/path)==expected for path,expected in protected_before.items())
assert protected_before==export_report['source_sha256']
assert all(sha(OUTPUT/name)==expected for name,expected in new_export_before.items())
assert sha(CSV_PATH)==export_report['csv_sha256']
assert export_report['all_passed']
print(f'PASS: all {len(arrays)} arrays and {sum(v.size for v in arrays.values()):,} original array elements reconstructed byte-identically.')
print('PASS: all decoded categorical, pitch, source and genre labels verified.')
print('PASS: original artifacts and the readable export are unchanged.')

,NPZ array,shape,dtype,byte-identical
0,inputs__basic_chord_ids,"[54, 8]",int64,True
1,inputs__basic_root_ids,"[54, 8]",int64,True
2,inputs__basic_quality_ids,"[54, 8]",int64,True
3,inputs__previous_basic_chord_ids,"[54, 8]",int64,True
4,inputs__next_basic_chord_ids,"[54, 8]",int64,True
5,inputs__key_ids,"[54, 8]",int64,True
6,inputs__mode_ids,"[54, 8]",int64,True
7,inputs__key_confidence_ids,"[54, 8]",int64,True
8,inputs__roman_ids,"[54, 8]",int64,True
9,inputs__harmonic_function_ids,"[54, 8]",int64,True


PASS: all 30 arrays and 25,218 original array elements reconstructed byte-identically.
PASS: all decoded categorical, pitch, source and genre labels verified.
PASS: original artifacts and the readable export are unchanged.


## Inspection result

The readable export preserves all model-ready inputs, targets and masks, plus
decoded labels and explicit sequence identities. It is a faithful inspection
view, not a replacement for the typed NPZ Dataset. Keep supervision and decoded
target columns separate from model inputs. No Transformer is implemented here.

In [8]:
display(counts)
print(f'{N} sequences × {L} positions = {len(records)} CSV rows.')
print(f'{int(real.sum())} real beats, {int((~real).sum())} padded positions; '
      f'{int(loss.sum())} eligible targets and {int((real & ~loss).sum())} ignored real targets.')
print('Padded positions in the final sequence:',np.flatnonzero(~real[-1]).tolist())
print('No numerical values were rounded in the CSV; all original dtypes and values are recoverable.')

,sequences,sequence_length,expected_rows_from_shape,CSV_rows,CSV_columns,real_positions,padded_positions,eligible_targets,invalid_real_targets,note_capacity
0,54,8,432,432,49,428,4,425,3,4


54 sequences × 8 positions = 432 CSV rows.
428 real beats, 4 padded positions; 425 eligible targets and 3 ignored real targets.
Padded positions in the final sequence: [4, 5, 6, 7]
No numerical values were rounded in the CSV; all original dtypes and values are recoverable.
